# The Statistics of Evals

A benchmark score is an **estimate**, computed from a finite sample, and it comes with an
uncertainty that is almost never reported. That omission is why so much of the published
model-comparison literature is unreliable: a 1-point difference on a 500-item benchmark
is well inside the noise, and treating it as a result means chasing effects that are not
there.

This notebook is the arithmetic that turns "model B scored higher" into "model B is
better, and here is how confident I am". None of it is difficult — it is the standard
statistics of proportions — but the details that matter for evals specifically (paired
comparisons, sampling variance, many benchmarks at once) are the ones most often skipped.

Follows [Eval Harness Design](eval-harness-design.ipynb); pairs with
[LLM-as-a-Judge](llm-as-judge.ipynb).

## 1. What & Why

Three distinct sources of uncertainty, and conflating them is the root of most mistakes:

1. **Item sampling.** Your benchmark is a sample of possible questions. Run the same
   model on a different 500 questions from the same distribution and the score moves.
   This is what a confidence interval on a proportion captures.
2. **Generation sampling.** At temperature > 0 the same model on the same item gives
   different answers. Re-running the identical eval produces a different number.
3. **Everything else** — prompt template, few-shot examples, harness version. Usually the
   largest of the three, and the hardest to quantify (see
   [Eval Harness Design](eval-harness-design.ipynb) Example 3).

The practical consequence: **before comparing two models, know your noise floor.** Run
the same model twice. Whatever difference you see is the floor, and any model comparison
smaller than it is not evidence of anything.

**Reach for this when** you are about to make a decision on an eval result — ship a
model, pick a training recipe, write a claim in a paper. **You can skip it** when the
difference is enormous and the direction is all you need; a 40-point gap does not need a
significance test.

## 2. Mental Model

**Polling.**

An eval is a poll of a population of questions. Everything pollsters know applies
directly:

- **Margin of error shrinks as `1/√n`.** To halve your error bar you need **four times**
  the items. This is the single most useful fact here, and it is why a 200-item benchmark
  cannot resolve small differences no matter how carefully it is built.
- **Ask the same people** and you can detect much smaller differences. A poll that
  re-interviews the same respondents (a *paired* design) is far more sensitive than two
  independent polls, because each respondent acts as their own control. In evals this is
  free: both models see the same items, so **always analyse paired**.
- **Ask enough questions and something will look significant.** Test twenty benchmarks at
  the 5% level and you expect one false positive even if nothing is different.

The one place the analogy breaks: a pollster cannot re-ask the same person under
identical conditions, but you *can* re-run a model. That gives you a direct measurement of
generation noise that pollsters would envy — and it is the measurement almost nobody
takes.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Confidence interval** | A range that would contain the true value in (say) 95% of repeated experiments. For a proportion, use Wilson — not the normal approximation. |
| **Wilson interval** | The proportion CI that stays correct at small `n` and extreme `p`. The normal ("Wald") interval fails badly there. |
| **Standard error** | `√(p(1−p)/n)` for a proportion. Shrinks as `1/√n`. |
| **Paired comparison** | Both models on the same items. Removes item difficulty as a source of variance and is far more powerful. |
| **McNemar's test** | The correct significance test for two models on the same binary-scored items. Uses only the *disagreements*. |
| **Statistical power** | Probability of detecting a real effect of a given size. Depends on effect size, `n`, and pairing. |
| **Effect size** | The magnitude of the difference. Significance without effect size is not a finding. |
| **Multiple comparisons** | Testing many hypotheses inflates false positives; correct with Bonferroni, Holm, or a false-discovery-rate procedure. |
| **Bootstrap** | Resample items with replacement to get a CI for any statistic, including ones with no closed form. |
| **Clustered items** | Items sharing a passage or template are not independent; naive CIs are too narrow. |
| **Variance from decoding** | Re-running at temperature > 0 changes the score. Report over several seeds, or use greedy decoding and say so. |

## 4. Setup

NumPy and the standard library. Everything below is a simulation with a known ground
truth, so each method can be checked against the answer it is supposed to recover — which
is the only honest way to demonstrate a statistical procedure.

In [1]:
# %pip install numpy

import math
import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — how big is the error bar, really?

The first question to ask of any benchmark number. Note how slowly it improves with `n`.

In [2]:
def wilson_interval(k, n, z=1.96):
    '''95% CI for a proportion. Correct at small n and extreme p, unlike the normal
    approximation, which can produce intervals extending past 0 or 1.'''
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))

def wald_interval(k, n, z=1.96):
    p = k / n
    half = z * math.sqrt(p * (1 - p) / n)
    return (p - half, p + half)

print("A model scoring 75% -- how wide is the interval?\n")
print(f"{'n items':>9} {'score':>7} {'95% CI (Wilson)':>22} {'+/- width':>11}")
for n in (50, 100, 200, 500, 1000, 5000, 20000):
    k = round(0.75 * n)
    lo, hi = wilson_interval(k, n)
    print(f"{n:9d} {k/n:7.1%} {f'[{lo:.1%}, {hi:.1%}]':>22} {(hi - lo) / 2:11.2%}")

print("\nTo halve the error bar you need FOUR times the items. A 500-item benchmark")
print("resolves differences of roughly +/-4 points; claiming a 2-point improvement")
print("from one is not supportable.\n")

print("Where the normal approximation breaks (small n, extreme p):")
print(f"{'k/n':>10} {'Wald (normal approx)':>26} {'Wilson':>22}")
for k, n in [(19, 20), (20, 20), (1, 30), (0, 50)]:
    w = wald_interval(k, n)
    ws = wilson_interval(k, n)
    print(f"{f'{k}/{n}':>10} {f'[{w[0]:.1%}, {w[1]:.1%}]':>26} {f'[{ws[0]:.1%}, {ws[1]:.1%}]':>22}")
print("\nThe Wald interval runs past 100% and collapses to zero width at k=0 or k=n --")
print("both nonsense. Use Wilson; it is four lines.")

A model scoring 75% -- how wide is the interval?

  n items   score        95% CI (Wilson)   +/- width
       50   76.0%         [62.6%, 85.7%]      11.56%
      100   75.0%         [65.7%, 82.5%]       8.38%
      200   75.0%         [68.6%, 80.5%]       5.96%
      500   75.0%         [71.0%, 78.6%]       3.79%
     1000   75.0%         [72.2%, 77.6%]       2.68%
     5000   75.0%         [73.8%, 76.2%]       1.20%
    20000   75.0%         [74.4%, 75.6%]       0.60%

To halve the error bar you need FOUR times the items. A 500-item benchmark
resolves differences of roughly +/-4 points; claiming a 2-point improvement
from one is not supportable.

Where the normal approximation breaks (small n, extreme p):
       k/n       Wald (normal approx)                 Wilson
     19/20            [85.4%, 104.6%]         [76.4%, 99.1%]
     20/20           [100.0%, 100.0%]        [83.9%, 100.0%]
      1/30              [-3.1%, 9.8%]          [0.6%, 16.7%]
      0/50               [0.0%, 0.0%]   

### Example 2 — pair your comparisons, and get a much smaller `n` for free

Both models answer the same items, so item difficulty is shared and can be cancelled out.
The paired analysis uses only the items where the models **disagree** — which is exactly
McNemar's test.

In [3]:
def run_pair(n_items, ability_a, ability_b, seed):
    '''Both models on the same items. Item difficulty is shared -- the whole point.'''
    r = np.random.default_rng(seed)
    difficulty = r.normal(0, 1.2, n_items)                # shared across models
    a = r.random(n_items) < 1 / (1 + np.exp(difficulty - ability_a))
    b = r.random(n_items) < 1 / (1 + np.exp(difficulty - ability_b))
    return a, b

def mcnemar_p(a, b):
    '''Two-sided McNemar via the exact binomial on the discordant pairs.'''
    n01 = int(np.sum(~a & b))       # only b correct
    n10 = int(np.sum(a & ~b))       # only a correct
    n = n01 + n10
    if n == 0:
        return 1.0, n01, n10
    k = min(n01, n10)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2**n
    return min(1.0, 2 * tail), n01, n10

def unpaired_p(a, b):
    '''Two-proportion z-test, pretending the two runs were independent samples.'''
    p1, p2 = a.mean(), b.mean()
    n1, n2 = len(a), len(b)
    p = (a.sum() + b.sum()) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    if se == 0:
        return 1.0
    z = (p2 - p1) / se
    return 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))

print(f"{'n items':>8} {'A':>7} {'B':>7} {'diff':>7} {'paired p':>10} {'unpaired p':>12} "
      f"{'discordant':>11}")
for n in (200, 500, 1000, 2000):
    a, b = run_pair(n, 0.30, 0.45, seed=n)
    p_pair, n01, n10 = mcnemar_p(a, b)
    print(f"{n:8d} {a.mean():7.1%} {b.mean():7.1%} {b.mean()-a.mean():+7.1%} "
          f"{p_pair:10.4f} {unpaired_p(a, b):12.4f} {f'{n01}/{n10}':>11}")

print("\nRead the two p-value columns where there is signal to find. At n=1000 the")
print("paired test reaches p<0.05 while the unpaired one does not, and at n=2000 it is")
print("an order of magnitude smaller. At n=200-500 neither detects anything and the two")
print("are indistinguishable -- pairing buys power, it does not manufacture evidence.")
print("\nThe mechanism: the paired test conditions on the items both models got right or")
print("both got wrong. Those carry no information about WHICH model is better, but they")
print("do inflate the variance of the unpaired estimate.")
print("\nBoth models see the same items in every real eval, so the paired analysis is")
print("always available and there is no reason to use the unpaired one.")

 n items       A       B    diff   paired p   unpaired p  discordant
     200   59.0%   61.5%   +2.5%     0.6254       0.6095       36/31
     500   57.2%   59.0%   +1.8%     0.5708       0.5641      104/95
    1000   57.1%   61.2%   +4.1%     0.0372       0.0622     205/164
    2000   54.0%   59.7%   +5.6%     0.0000       0.0003     426/314

Read the two p-value columns where there is signal to find. At n=1000 the
paired test reaches p<0.05 while the unpaired one does not, and at n=2000 it is
an order of magnitude smaller. At n=200-500 neither detects anything and the two
are indistinguishable -- pairing buys power, it does not manufacture evidence.

The mechanism: the paired test conditions on the items both models got right or
both got wrong. Those carry no information about WHICH model is better, but they
do inflate the variance of the unpaired estimate.

Both models see the same items in every real eval, so the paired analysis is
always available and there is no reason to use the

### Example 3 — how many items do you actually need?

Power analysis, by simulation. This is the calculation to run *before* building a
benchmark, not after being disappointed by it.

In [4]:
def power(n_items, gap, trials=400, alpha=0.05, base=0.30):
    '''Fraction of experiments that detect a true `gap` in ability, paired analysis.'''
    hits = 0
    for t in range(trials):
        a, b = run_pair(n_items, base, base + gap, seed=10_000 + t)
        if mcnemar_p(a, b)[0] < alpha:
            hits += 1
    return hits / trials

print("Power to detect a true difference (paired test, alpha = 0.05).")
print("80% power is the usual target. Note `gap` is in ABILITY (logit) units:")
print("with this item-difficulty spread, gap 0.15 is worth roughly 2-4 accuracy")
print("points and gap 0.30 roughly 5-6 -- so even the largest column here is a")
print("difference you would describe as modest.\n")
print(f"{'n items':>8} " + " ".join(f"{'gap ' + str(g):>10}" for g in (0.05, 0.15, 0.30)))
for n in (100, 250, 500, 1000, 2000):
    row = " ".join(f"{power(n, g):10.0%}" for g in (0.05, 0.15, 0.30))
    print(f"{n:8d} " + row)

print("\nA small true difference simply cannot be resolved by a small benchmark, at any")
print("level of care in the harness. A ~5-point improvement needs about 1000 items to")
print("be detected reliably; a 2-point one is out of reach for every size shown here.")
print("Running the experiment anyway does not give you a weak answer -- it gives you a")
print("coin flip that occasionally reports 'significant'.")

print("\nFalse-positive check (gap = 0, so every 'detection' is spurious):")
for n in (250, 1000):
    print(f"  n={n:5d}  detections at alpha=0.05: {power(n, 0.0):.1%}  (should be ~5%)")

Power to detect a true difference (paired test, alpha = 0.05).
80% power is the usual target. Note `gap` is in ABILITY (logit) units:
with this item-difficulty spread, gap 0.15 is worth roughly 2-4 accuracy
points and gap 0.30 roughly 5-6 -- so even the largest column here is a
difference you would describe as modest.

 n items   gap 0.05   gap 0.15    gap 0.3
     100         4%         6%        11%
     250         5%        10%        30%
     500         5%        17%        52%


    1000         6%        28%        81%


    2000        10%        53%        98%

A small true difference simply cannot be resolved by a small benchmark, at any
level of care in the harness. A ~5-point improvement needs about 1000 items to
be detected reliably; a 2-point one is out of reach for every size shown here.
Running the experiment anyway does not give you a weak answer -- it gives you a
coin flip that occasionally reports 'significant'.

False-positive check (gap = 0, so every 'detection' is spurious):
  n=  250  detections at alpha=0.05: 4.2%  (should be ~5%)
  n= 1000  detections at alpha=0.05: 4.0%  (should be ~5%)


### Example 4 — twenty benchmarks, one false positive

You evaluate a new recipe on a suite. Nothing has actually changed. How often does at
least one benchmark look significant?

In [5]:
def suite_run(n_benchmarks, n_items, gap, seed):
    return [mcnemar_p(*run_pair(n_items, 0.30, 0.30 + gap, seed=seed * 1000 + i))[0]
            for i in range(n_benchmarks)]

def holm(pvals, alpha=0.05):
    '''Holm-Bonferroni: uniformly more powerful than plain Bonferroni, same guarantee.'''
    order = sorted(range(len(pvals)), key=lambda i: pvals[i])
    m = len(pvals)
    rejected = [False] * m
    for rank, i in enumerate(order):
        if pvals[i] <= alpha / (m - rank):
            rejected[i] = True
        else:
            break
    return rejected

M = 20
trials = 300
any_naive = any_bonf = any_holm = 0
for t in range(trials):
    ps = suite_run(M, 500, gap=0.0, seed=t)          # NOTHING is really different
    any_naive += any(p < 0.05 for p in ps)
    any_bonf += any(p < 0.05 / M for p in ps)
    any_holm += any(holm(ps))

print(f"{M} benchmarks, no real difference anywhere, {trials} repeats of the whole suite:")
print(f"  at least one 'significant' at p<0.05      : {any_naive/trials:.0%}")
print(f"  ... with Bonferroni correction (p<0.05/{M}) : {any_bonf/trials:.0%}")
print(f"  ... with Holm-Bonferroni                   : {any_holm/trials:.0%}")

print("\nUncorrected, you will find a 'result' in most sweeps where nothing changed.")
print("This is the mechanism behind a great deal of irreproducible model comparison:")
print("run enough benchmarks, report the ones that moved.")
print("\nCorrect for the number of tests, or -- better -- decide which benchmark is your")
print("primary metric BEFORE running, and treat the rest as exploratory.")

20 benchmarks, no real difference anywhere, 300 repeats of the whole suite:
  at least one 'significant' at p<0.05      : 61%
  ... with Bonferroni correction (p<0.05/20) : 4%
  ... with Holm-Bonferroni                   : 4%

Uncorrected, you will find a 'result' in most sweeps where nothing changed.
This is the mechanism behind a great deal of irreproducible model comparison:
run enough benchmarks, report the ones that moved.

Correct for the number of tests, or -- better -- decide which benchmark is your
primary metric BEFORE running, and treat the rest as exploratory.


### Example 5 — the noise floor you get for free by re-running

Item-sampling error is only part of it. At temperature > 0, the same model on the same
items gives a different score every run.

In [6]:
def eval_run(n_items, ability, temperature, seed):
    '''Higher temperature -> answers drift toward chance, so the score falls AND each
    run is an independent draw rather than a repeatable measurement.'''
    r = np.random.default_rng(seed)
    difficulty = np.random.default_rng(999).normal(0, 1.2, n_items)   # SAME items always
    p = 1 / (1 + np.exp(difficulty - ability))
    p = (1 - temperature) * p + temperature * 0.25       # blend toward 25% chance
    return float(np.mean(r.random(n_items) < p))

N = 1000
print(f"the SAME model on the SAME {N} items, 8 times over:\n")
print(f"{'temperature':>12} {'runs':>52} {'spread':>8}")
for temp in (0.0, 0.3, 0.7, 1.0):
    runs = [eval_run(N, 0.4, temp, seed=s) for s in range(8)]
    if temp == 0.0:
        runs = [eval_run(N, 0.4, 0.0, seed=0)] * 8      # greedy decoding is deterministic
    spread = max(runs) - min(runs)
    print(f"{temp:12.1f} {' '.join(f'{r:.1%}' for r in runs):>52} {spread:8.1%}")

print("\nAt temperature 0.7 the same model spans a range comparable to the differences")
print("people routinely report between model versions. A single run at temperature > 0")
print("is one draw from that distribution.")
print("\nTwo defensible options, and one indefensible one:")
print("  - greedy decoding (temperature 0), reported as such: reproducible")
print("  - several seeds, reporting mean and spread: honest about the variance")
print("  - one run at temperature 0.7, reported as 'the score': not a measurement")

the SAME model on the SAME 1000 items, 8 times over:

 temperature                                                 runs   spread
         0.0      58.7% 58.7% 58.7% 58.7% 58.7% 58.7% 58.7% 58.7%     0.0%
         0.3      46.7% 46.8% 49.2% 48.3% 46.9% 49.0% 48.6% 49.7%     3.0%
         0.7      31.9% 34.9% 33.9% 35.0% 32.4% 35.0% 34.2% 34.5%     3.1%
         1.0      23.1% 23.9% 25.7% 25.3% 24.1% 26.7% 24.1% 25.7%     3.6%

At temperature 0.7 the same model spans a range comparable to the differences
people routinely report between model versions. A single run at temperature > 0
is one draw from that distribution.

Two defensible options, and one indefensible one:
  - greedy decoding (temperature 0), reported as such: reproducible
  - several seeds, reporting mean and spread: honest about the variance
  - one run at temperature 0.7, reported as 'the score': not a measurement


## 6. Gotchas & Pitfalls

- **Reporting a score with no interval.** Example 1. A bare number invites the reader to
  treat noise as signal.
- **The normal-approximation interval at small `n` or extreme `p`.** Example 1: it
  produces intervals extending past 100% and zero-width intervals at `k=0`. Use Wilson.
- **Unpaired analysis of paired data.** Example 2 — you are discarding most of your
  statistical power for nothing.
- **No power analysis before building the benchmark.** Example 3. If 250 items cannot
  resolve the effect you care about, that is worth knowing before you spend the money.
- **Running twenty benchmarks and reporting the winners.** Example 4. Pre-register a
  primary metric, or correct for the number of tests.
- **Ignoring generation variance.** Example 5. At temperature > 0, one run is a sample.
- **Treating clustered items as independent.** Twenty questions about one passage are
  not twenty independent observations; naive CIs are too narrow. Bootstrap by *cluster*.
- **Significance without effect size.** With 100k items a 0.1-point difference is highly
  significant and completely uninteresting. Report both.
- **Optimising against the test set.** Every decision made by looking at a benchmark
  spends a little of its validity, and no statistic corrects for that.
- **Comparing across harness versions.** The CI covers item sampling, not the fact that
  you changed the parser in between.

## 7. When to Use vs Alternatives

| Question | Tool |
|---|---|
| How precise is this score? | **Wilson interval** on the proportion |
| Is model B better than A on the same items? | **McNemar's test** (paired) |
| How many items do I need? | **Power simulation** (Example 3) |
| Many benchmarks at once | **Holm-Bonferroni**, or pre-register one primary metric |
| A statistic with no closed form (macro-F1, pass@k, mean judge score) | **Bootstrap** over items |
| Items grouped by passage/template | **Cluster bootstrap** — resample groups, not items |
| Is the harness itself stable? | Re-run the same model (Example 5); that is your floor |

**The honest position.** The single highest-value habit is not any particular test — it
is **running the same model twice and looking at the difference**. That one number tells
you which comparisons are worth making at all, costs one extra eval run, and is skipped
almost universally.

After that, the ordering is: report intervals, analyse paired, and decide your primary
metric before you look. None of it is sophisticated statistics; it is the standard
machinery for proportions, applied consistently.

## 8. Resources

- [Interval Estimation for a Binomial Proportion](https://projecteuclid.org/euclid.ss/1009213286) — Brown, Cai & DasGupta; the definitive treatment of why the Wald interval fails and Wilson does not.
- [Wilson score interval](https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval#Wilson_score_interval) — the formula implemented in Example 1.
- [McNemar's test](https://en.wikipedia.org/wiki/McNemar%27s_test) — the paired test of Example 2, including when to prefer the exact binomial form.
- [The Hitchhiker's Guide to Testing Statistical Significance in Natural Language Processing](https://aclanthology.org/P18-1128/) — a practical decision procedure for choosing a test in NLP settings.
- [Adding Error Bars to Evals](https://arxiv.org/abs/2411.00640) — Anthropic, 2024; exactly this notebook's subject applied to modern LLM evaluations, including clustered items and paired designs.
- [We need to talk about random seeds](https://arxiv.org/abs/2210.13393) — how much of a reported improvement is seed variance.
- [Controlling the False Discovery Rate](https://www.jstor.org/stable/2346101) — Benjamini & Hochberg; the less conservative alternative to Bonferroni when testing many benchmarks.